# Spark Tuning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## How should you approach Spark tuning in PySpark?

Spark tuning should start with evidence, not random configuration changes. The usual workflow is:

1. Reproduce the slow job with representative data.
2. Read the Spark UI and `explain("formatted")`.
3. Identify the bottleneck: scan, shuffle, skew, spill, join, small files, driver memory, or sink write.
4. Change one thing.
5. Measure again.

Useful first checks:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
df.explain("formatted")
print("partitions:", df.rdd.getNumPartitions())
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Real-world example: a job that reads 4 TB and then filters to one day may be slow because partition pruning is not working. Increasing executor memory will not fix that. You need to fix table partitioning, predicates, or file layout.

Under the hood, a DataFrame program is not executed as written. Spark builds a logical plan, passes it to the Catalyst optimizer, then chooses a physical plan by estimated cost. That physical plan becomes a DAG of RDD stages, and a new stage starts wherever data has to be exchanged across the network. The stage is the real unit of tuning: inside a stage Spark runs one task per partition, all independent, and the stage cannot finish until its slowest task finishes. That single rule explains most Spark slowness. A stage with 1,000 balanced tasks and one straggler is as slow as the straggler, no matter how much memory you add.


Worked example. A job reads 4 TB, filters to 200 GB, and groups by `order_date`. The scan is I/O over as many tasks as there are files; the aggregate is a shuffle of roughly 200 GB. With the default `spark.sql.shuffle.partitions = 200`, each reducer handles about 1 GB and spills. Raising it to 2,000 drops each reducer to about 100 MB, which fits. Notice these are two independent numbers: scan parallelism is set by file layout, shuffle parallelism by `spark.sql.shuffle.partitions` and AQE.


The common mistake is to reach for executor memory first. More memory only helps when the Spark UI shows spill, high GC time, or a killed container. If a stage is dominated by one slow task, or the plan never prunes partitions, extra memory changes nothing but the bill.


In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("orders-tuning-baseline")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

# Record the starting configuration so an experiment is reproducible.
baseline = {
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "adaptive_enabled": spark.conf.get("spark.sql.adaptive.enabled"),
    "broadcast_threshold": spark.conf.get("spark.sql.autoBroadcastJoinThreshold"),
}
print(baseline)


## How do you read a Spark physical plan?

The physical plan shows how Spark will execute the query. It helps you identify scans, filters, exchanges, joins, aggregations, and code generation stages.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.table("main.gold.fact_order")
customers = spark.table("main.gold.dim_customer")

result = (
    orders
    .where(F.col("order_date") >= F.lit("2026-10-01"))
    .join(customers, "customer_id")
    .groupBy("customer_segment")
    .agg(F.sum("amount").alias("revenue"))
)

result.explain("formatted")

Look for:

+ `FileScan`: what files and columns Spark reads
+ `PushedFilters`: predicates pushed to the reader
+ `PartitionFilters`: partition pruning
+ `Exchange`: shuffle boundary
+ `BroadcastHashJoin`: small side was broadcast
+ `SortMergeJoin`: both sides were shuffled and sorted
+ `AdaptiveSparkPlan`: AQE is enabled

Interview point: if you cannot explain the plan at a high level, you are tuning blind.

Catalyst runs four phases. Analysis resolves column names and types against the catalog. Logical optimization rewrites the tree with rules such as predicate pushdown, constant folding, and column pruning. Physical planning generates candidate plans and picks the cheapest using statistics from the metastore and from file metadata. Finally whole-stage code generation fuses operators into a single generated Java function, which is why `WholeStageCodegen` appears in the plan. `explain("formatted")` prints the chosen plan; `explain("cost")` prints the candidate plans with their estimated row counts and sizes, which is how you see why Spark chose one join over another.


A plan for a filtered join and aggregate reads like this:


```text
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_segment], functions=[sum(amount)])
   +- Exchange hashpartitioning(customer_segment, 200)
      +- Project [customer_segment, amount]
         +- BroadcastHashJoin [customer_id], [customer_id], Inner
            :- Filter (isnotnull(order_date) AND (order_date >= 2026-10-01))
            :  +- FileScan parquet fact_order[customer_id, amount, order_date]
            :        PartitionFilters: [order_date >= 2026-10-01]
            +- BroadcastExchange ...
```

Read it bottom-up. `PartitionFilters` proves the date predicate pruned directories. `FileScan parquet fact_order[customer_id, amount, order_date]` shows only three columns were projected, so the other column chunks are never read. `BroadcastHashJoin` means the join itself needed no shuffle. The single `Exchange hashpartitioning(customer_segment, 200)` is the only network boundary, created by the `groupBy`.


The common mistake is reading the top of the plan as the expensive part. The plan is a tree and the leaves feed everything; the cost usually sits in the scans, the exchanges, and the joins, not in the printed root. A second trap: `AdaptiveSparkPlan isFinalPlan=false` means this is a pre-execution guess and AQE may replace the join and the partition counts at runtime. Always confirm with the final plan or the UI stage metrics.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# cost-based view: competing physical plans with estimated sizes
result.explain("cost")

# formatted view: the plan that is actually chosen, operator by operator
result.explain("formatted")


## What is a shuffle and why is it expensive?

A shuffle redistributes data across executors. It usually happens when Spark needs all records with the same key in the same partition.

Common shuffle operations:

+ `groupBy`
+ `distinct`
+ `dropDuplicates`
+ `join` without broadcast
+ `repartition`
+ Window functions with `partitionBy`

Example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
daily_revenue = (
    spark.table("main.silver.orders")
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)

This groups all records for the same `order_date` together. If one date has much more data than others, one shuffle partition can become much larger and slower.

How to inspect shuffle symptoms:

+ Spark UI stage has high shuffle read/write.
+ A few tasks are much slower than the rest.
+ Tasks spill to disk.
+ Executor logs show fetch failures or memory pressure.

Mechanically, a shuffle has a map side and a reduce side. Each map task partitions its output by `hash(key) % numPartitions`, sorts it, and writes one shuffle block per reducer to local disk. Reducers then fetch their blocks across the network from every mapper, merge them, and continue. Every one of those steps costs something: serialization to bytes, a disk write and read, a network transfer, decompression, and a sort or hash build on the reduce side. Fetch failures during this phase force Spark to recompute and resend the lost map outputs, which is why a failed shuffle stage can be much more expensive than its first attempt suggested.


Worked example. Moving 500 GB through a shuffle at an aggregate 1 GB/s of network and disk bandwidth costs at least 500 seconds of pure transfer, and the real number includes serialization and sorting on top. Replace that with a broadcast of a 20 MB dimension and the shuffle disappears entirely: the same join streams the large side and probes an in-memory hash map, so there is no reduce-side disk I/O at all.


The common mistake is assuming a shuffle is free because the code is short. `df.orderBy(...)` before a write triggers a range shuffle. `df.distinct()` or `dropDuplicates()` is a shuffle even when the duplicate rate is high. `F.countDistinct` on a high-cardinality column shuffles every distinct value. And `repartition()` with no arguments performs a full round-robin shuffle for no benefit.


## How do you choose the right number of shuffle partitions?

`spark.sql.shuffle.partitions` controls the default number of partitions after many shuffle operations. The default is often `200`, which may be too low for huge jobs or too high for small jobs.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.shuffle.partitions", "800")

Rule of thumb:

+ Too few partitions: large tasks, memory spills, slow reducers
+ Too many partitions: many tiny tasks, scheduling overhead, small output files
+ Better target: partitions large enough to reduce overhead, small enough to avoid spills

Example for a 2 TB aggregation:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.shuffle.partitions", "2000")

agg = (
    spark.table("main.silver.events")
    .groupBy("event_date", "event_type")
    .agg(F.count("*").alias("events"))
)

On Databricks or Spark 3+, enable AQE so Spark can coalesce shuffle partitions at runtime:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

Interview answer: set a reasonable starting value, use AQE, then validate in the Spark UI by checking task sizes, spills, and scheduling overhead.

The right number comes from a target partition size, not from a guess. A shuffle partition should hold roughly 100 to 200 MB: large enough that task scheduling and per-task overhead are negligible, small enough that a sort or hash aggregate for that partition fits in execution memory. The arithmetic is `partitions = ceil(shuffle_bytes / target_size)`. With AQE enabled, `spark.sql.shuffle.partitions` sets the starting count and `spark.sql.adaptive.advisoryPartitionSizeInBytes` (default 64 MB) is the size AQE coalesces toward after it sees the real partition sizes.


Worked example. A 2 TB shuffle with the default 64 MB advisory size implies about 32,000 target partitions. Raise the advisory size to 128 MB and the same shuffle targets about 16,000. That sounds like a lot, but each one is a small task on a small partition, and AQE will merge any that turn out to be tiny. The failure mode at the other extreme is 200 partitions of 10 GB each, where every reducer spills repeatedly.


The common mistake is to set `spark.sql.shuffle.partitions` to a large round number "just in case" and leave AQE off. That produces tens of thousands of tiny tasks, each with its own scheduling overhead, plus tens of thousands of tiny output files. A second mistake is changing the setting after an action has already run: the number is read when the shuffle is planned, so it applies to later actions only.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Starting count for the shuffle, then let AQE coalesce toward 128 MB.
spark.conf.set("spark.sql.shuffle.partitions", "4000")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")


## How do repartition and coalesce differ in PySpark?

`repartition` increases or decreases partitions with a full shuffle. `coalesce` usually reduces partitions without a full shuffle.

Use `repartition` when you need an even redistribution:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
balanced = orders.repartition(400)

Use `coalesce` when reducing partitions after filtering:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
small_result = (
    orders
    .where("order_date = date '2026-10-04'")
    .coalesce(24)
)

Key difference:

+ `repartition(400)`: expensive, but balances data across 400 partitions
+ `coalesce(24)`: cheaper, but can preserve imbalance from upstream partitions

Real-world example: if a job filters a month of data down to one small day, `coalesce` before writing can reduce small files. If partitions are badly skewed, `repartition` may be better despite the shuffle.

`repartition` builds a new RDD with a full shuffle and uses either hash partitioning (when you pass columns) or round-robin partitioning (when you pass only a count). It can increase or decrease the partition count and it balances data by construction. `coalesce` does something much narrower: it merges existing partitions into a smaller number by combining adjacent partitions that already sit on the same executor, so it creates a narrow dependency and no shuffle. It cannot increase the partition count. `coalesce(n, shuffle=True)` is a repartition in disguise.


Worked example. After a filter, 1,000 partitions hold 4 GB in total. `coalesce(16)` merges them without a shuffle, so each downstream task gets about 256 MB, which is above the comfortable range and will likely spill. `repartition(16)` pays one shuffle but produces 16 even partitions. When the upstream data is skewed, coalesce can hand one task several gigabytes because it merges whole partitions and never splits them.


The common mistake is `coalesce(1)` on a large result to "avoid small files". It collapses all the work into a single task and one core, which can turn a two-minute write into an hour. The mirror-image mistake is calling `repartition()` with no arguments when the goal was just to reduce the output partition count; that forces an unnecessary full shuffle.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# reduce output partitions cheaply after a selective filter
filtered = orders.where("order_date = date '2026-10-04'").coalesce(24)

# balance a skewed frame before a downstream join; this pays a shuffle
balanced = orders.repartition(400, "customer_id")

print("filtered partitions:", filtered.rdd.getNumPartitions())
print("balanced partitions:", balanced.rdd.getNumPartitions())


## How do you use repartition by column?

Column-based repartitioning puts rows with the same key into the same partition. This can help before joins, aggregations, or table writes, but it creates a shuffle.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_by_customer = orders.repartition(800, "customer_id")

Useful before a key-based operation:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
deduped = (
    spark.table("main.bronze.orders_cdc")
    .repartition(800, "order_id")
    .dropDuplicates(["order_id", "source_lsn"])
)

Useful before partitioned writes:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    daily_orders
    .repartition("order_date")
    .write
    .mode("append")
    .partitionBy("order_date")
    .saveAsTable("main.silver.orders")
)

Be careful: repartitioning by a low-cardinality column such as `country_code` can create skew. If 80 percent of rows are `US`, the `US` partition can dominate the job.

`repartition(n, col)` applies `HashPartitioning` to the column: `partition = pmod(hash(value), n)`. Rows with the same value always land in the same partition, which is exactly what a keyed join, aggregation, or dedup needs. `repartition(col)` without a count uses `spark.sql.shuffle.partitions`. Two properties matter in practice: hashing destroys any ordering that existed, and null keys are all hashed to the same partition, so a large null group becomes a skew problem.


Worked example before a partitioned write. If you `repartition("order_date")` and then write with `partitionBy("order_date")`, each Spark partition mostly holds one date and writes a small number of files into that date's directory. If you skip the repartition and write from 1,000 arbitrary partitions, each date directory receives one file per Spark partition that contains that date, which is how a single day ends up with hundreds of small files.


The common mistake is repartitioning by one column and then shuffling again on a different key immediately after, which pays for two exchanges where one was needed. Another is repartitioning by a low-cardinality column such as `country_code`: the distinct values are fewer than the requested partition count, so most partitions stay empty while the largest value dominates.


## How do you fix data skew with salting?

Data skew happens when a few keys have much more data than other keys. Salting spreads a hot key across multiple artificial buckets so Spark can process it in parallel.

Skewed join example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders")
promotions = spark.table("main.silver.customer_promotions")

joined = orders.join(promotions, "customer_id")

If `customer_id = 'guest'` appears in hundreds of millions of rows, one join partition becomes huge.

Salt the large side:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

salt_buckets = 32

orders_salted = orders.withColumn(
    "salt",
    (F.rand(seed=42) * salt_buckets).cast("int"),
)

Replicate the small side across salts:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
salts = spark.range(salt_buckets).withColumnRenamed("id", "salt")

promotions_salted = promotions.crossJoin(salts)

Join on the original key plus salt:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
joined = (
    orders_salted
    .join(promotions_salted, ["customer_id", "salt"], "left")
    .drop("salt")
)

When to use salting:

+ One or a few keys dominate a join or aggregation.
+ Broadcast join is not possible.
+ AQE skew join handling is not enough.

When not to use salting:

+ The small side is not actually small enough to replicate.
+ The key distribution is already even.
+ You have not verified skew in the Spark UI.

Aggregation salting has two phases:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
salted = orders.withColumn("salt", (F.rand(seed=7) * 32).cast("int"))

partial = (
    salted
    .groupBy("customer_id", "salt")
    .agg(F.sum("amount").alias("partial_amount"))
)

final = (
    partial
    .groupBy("customer_id")
    .agg(F.sum("partial_amount").alias("total_amount"))
)

The first aggregation spreads hot keys. The second aggregation removes the salt and produces the final result.

There are two ways to salt. The widely used one adds `F.rand()` to the large side and cross joins the small side with a range of salts, which works when the small side is cheap to replicate but requires that replication. The more efficient one is deterministic: both sides compute `salt = pmod(xxhash64(key), n)`, join on `(key, salt)`, and no replication is needed. The deterministic version is reproducible, so a rerun produces the same partitioning, which matters when you are debugging a job.


Worked example. A fact table has one billion rows and a single customer key holds 40 percent of them. That is 400 million rows in one join partition. With 32 salts the hot key splits into 32 groups of 12.5 million, so the partition shrinks by 32x. The cost is on the other side: the small relation is now replicated 32 times, so a 20 MB dimension becomes 640 MB and may no longer fit the broadcast budget.


The common mistake is salting before confirming skew. Salting adds a shuffle and a wider join key, and it does nothing if the distribution was already even. Two more: forgetting to drop the salt column before writing the result, and using `F.rand()` on only one side, which makes the salted key never match.


In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

salt_buckets = 32

# Deterministic salt: both sides can compute it without replicating the dimension.
orders_salted = orders.withColumn(
    "salt", F.pmod(F.xxhash64("customer_id"), F.lit(salt_buckets))
)
promotions_salted = promotions.withColumn(
    "salt", F.pmod(F.xxhash64("customer_id"), F.lit(salt_buckets))
)

joined = orders_salted.join(promotions_salted, ["customer_id", "salt"]).drop("salt")


## How do you tune joins in PySpark?

Join tuning starts with the size and distribution of both sides.

Typical decisions:

+ Broadcast the small side if it safely fits in executor memory.
+ Repartition both large sides by the join key if both sides are large.
+ Filter and select columns before joining.
+ Deduplicate dimensions before joining to avoid row multiplication.
+ Use salting or AQE skew handling for hot keys.

Good join hygiene:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = (
    spark.table("main.silver.orders")
    .where("order_date >= date '2026-10-01'")
    .select("order_id", "customer_id", "amount")
)

customers = (
    spark.table("main.gold.dim_customer")
    .where("is_current = true")
    .select("customer_id", "customer_segment")
    .dropDuplicates(["customer_id"])
)

joined = orders.join(customers, "customer_id", "left")

Bad pattern:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
joined = spark.table("main.silver.orders").join(
    spark.table("main.gold.dim_customer"),
    "customer_id",
)

The bad pattern reads unnecessary columns, may include historical dimension rows, and may multiply facts if the dimension is not unique by `customer_id`.

Spark picks among join strategies by relative size. `BroadcastHashJoin` collects the small side to every executor and builds a hash map, so the large side is never shuffled. `SortMergeJoin` shuffles and sorts both sides by the join key and streams the merge, which scales to two large inputs and spills gracefully. `ShuffledHashJoin` shuffles both sides but builds a hash map instead of sorting, and is controlled by `spark.sql.join.preferSortMergeJoin` and AQE. Non-equi joins fall back to `BroadcastNestedLoopJoin` or a Cartesian product, which is almost always the wrong plan.


Worked example. A 1 TB fact joined to a 5 GB dimension must use a sort-merge join: 5 GB cannot be broadcast, and the shuffle of the fact side is unavoidable. Change the dimension to a 20 MB lookup table and Spark broadcasts instead, and the 1 TB shuffle disappears. The decision is entirely about the size of the small side, which is why stale table statistics cause bad plans.


The common mistake is joining before filtering and selecting columns, which shuffles data you were about to discard. Another is joining on an expression such as `F.lower(a.customer_id) == F.lower(b.customer_id)`: that is not an equi-join on a column, so Spark cannot use a hash or sort-merge join. The third is a dimension that is not unique on the key, which multiplies fact rows and can silently double a revenue number.


## How does broadcast join tuning work?

A broadcast join sends a small DataFrame to every executor, avoiding a shuffle of the large side.

Explicit broadcast:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql.functions import broadcast

orders = spark.table("main.silver.orders")
calendar = spark.table("main.ref.calendar")

joined = orders.join(broadcast(calendar), "order_date", "left")

Configure automatic broadcast threshold:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)

Disable auto broadcast for testing:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

Use broadcast when:

+ The small side comfortably fits in executor memory.
+ The same dimension is joined to a much larger fact table.
+ You want to avoid a sort-merge join shuffle.

Avoid broadcast when:

+ The dimension is large or underestimated.
+ Executors have tight memory.
+ The broadcast side is produced by an expensive upstream query that will be recomputed.

Validate with:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
joined.explain("formatted")

Look for `BroadcastHashJoin` and `BroadcastExchange`.

At runtime, a broadcast join executes a `BroadcastExchange`: the small relation is collected to the driver and then served to executors as `TorrentBroadcast` blocks, where peers share pieces with each other rather than all pulling from the driver. The automatic decision uses the optimizer's estimated size for the relation compared against `spark.sql.autoBroadcastJoinThreshold`, default 10 MB. That estimate can be wrong for views, for relations whose statistics are stale, and for data that was just written. The real costs are driver memory to hold the collected relation and startup latency while every executor receives it.


Worked example. A product dimension has 10 million rows at roughly 100 bytes each, so about 1 GB. That is 100x the default threshold, so Spark chooses a sort-merge join and shuffles the fact side. If someone adds a `broadcast()` hint because the table is called "dim", the driver must hold 1 GB, each executor must hold 1 GB, and the job will probably fail with a driver or executor out-of-memory error instead of running slowly.


The common mistake is forcing a broadcast because the plan "looks better" without checking the size. Another is raising the threshold globally to 1 GB to make one join broadcast, which then quietly broadcasts other relations that were never meant to be. A third is broadcasting a relation produced by an expensive subquery, which recomputes it on every executor rather than once.


In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

joined = events.join(F.broadcast(products), "product_id")
joined.explain("formatted")  # confirm BroadcastHashJoin / BroadcastExchange

# make the automatic decision explicit and visible
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024 * 1024))


## How does Adaptive Query Execution help performance?

Adaptive Query Execution, or AQE, lets Spark adjust the physical plan during execution based on runtime statistics.

Enable AQE:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

AQE can:

+ Coalesce small shuffle partitions
+ Split skewed shuffle partitions
+ Switch join strategy at runtime
+ Avoid over-partitioning small outputs

Example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.shuffle.partitions", "2000")
spark.conf.set("spark.sql.adaptive.enabled", "true")

result = (
    spark.table("main.silver.orders")
    .groupBy("order_date", "status")
    .agg(F.count("*").alias("orders"))
)

Without AQE, Spark may write 2,000 tiny shuffle partitions. With AQE, Spark can coalesce them after seeing runtime sizes.

Interview point: AQE is helpful, but it does not replace good table design, filtering, join hygiene, and skew analysis.

AQE re-plans between query stages. Spark cuts the plan at shuffle boundaries; when the map side of a shuffle finishes, it has exact partition sizes and row counts instead of estimates. It then re-optimizes the rest of the plan: coalescing many small post-shuffle partitions into fewer larger ones (`spark.sql.adaptive.coalescePartitions.enabled`), splitting partitions that are far above the median into smaller ones (`spark.sql.adaptive.skewJoin.enabled`), switching a sort-merge join to a broadcast join when the runtime size turns out small, and reading shuffle data locally where possible (`spark.sql.adaptive.localShuffleReader.enabled`). Since Spark 3.2 AQE is on by default.


Worked example. `spark.sql.shuffle.partitions` is left at 200 and the shuffle produces 200 partitions of wildly different sizes: 190 of them are about 5 MB and ten are about 2 GB. Without AQE you pay for 200 tasks and ten that spill. With AQE, the 190 small partitions coalesce toward the 64 MB advisory size and the large ones are split, so the same job runs with fewer, more even tasks and no spill.


The common mistake is expecting AQE to fix everything. It only acts at shuffle boundaries, so a narrow-stage skew (for example one huge partition produced by a `map` that never shuffles) is untouched. It also cannot recover from a bad partition key or a scan that reads every file. And `AdaptiveSparkPlan isFinalPlan=false` in the plan output means you are looking at the pre-execution plan, not what ran.


## When should you cache or persist DataFrames?

Cache only when the same DataFrame is reused multiple times and is expensive to recompute.

Good cache use:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark import StorageLevel

base = (
    spark.table("main.silver.orders")
    .where("order_date >= date '2026-10-01'")
    .select("order_id", "customer_id", "amount", "order_date")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

high_value = base.where("amount >= 1000").count()
daily = base.groupBy("order_date").agg(F.sum("amount").alias("revenue"))

base.unpersist()

Bad cache use:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
df = spark.table("main.silver.orders").cache()
df.write.mode("overwrite").saveAsTable("main.tmp.once")

If the DataFrame is used once, caching adds overhead. If it is too large for memory, it may evict useful data or spill.

Interview point: caching is a performance tool, not a default habit. Cache after filters and column pruning, materialize it with an action, and unpersist when done.

`persist` and `cache` are lazy: nothing is stored until an action runs, and storage happens per partition on the executors that computed it. `MEMORY_ONLY` keeps partitions as deserialized Java objects, which is the fastest to read but the largest in memory; any partition that does not fit is evicted and recomputed from lineage on next use. `MEMORY_AND_DISK` spills evicted partitions to local disk instead of dropping them. The `_SER` variants store serialized bytes, which are much smaller but cost CPU on every read. `DISK_ONLY` always writes to disk. Memory storage competes with execution memory inside the unified region governed by `spark.memory.fraction` and `spark.memory.storageFraction`, so a big shuffle can evict cached blocks.


Worked example. A 100 GB DataFrame cached across 20 executors with 16 GB heaps each gives 320 GB of raw heap, so on paper it fits. In practice the unified region is only 0.6 of the heap after the 300 MB reserved area, so real storage capacity is roughly 0.6 x 15.7 GB x 20, about 188 GB, and the storage share of that is half. Caching this frame will evict constantly and recompute more than it saves.


The common mistake is caching as a reflex. Cache only a DataFrame that is reused more than once, that is expensive to recompute, and that fits. Cache after filtering and column pruning, trigger it with a cheap action, and call `unpersist()` when the reuse is over. Caching a frame you read once adds memory pressure and a shuffle-free recompute cost with no benefit.


## How do you tune file sizes and avoid small files?

Small files slow down Spark because each file creates metadata overhead and often a tiny task. They also hurt table formats because metadata grows.

Small-file symptoms:

+ Thousands of files under a small partition
+ Spark UI shows many tiny input tasks
+ Queries spend time listing files and scheduling tasks
+ Downstream reads are slow even when data volume is modest

Control output partitions before writing:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    daily_orders
    .repartition(96, "order_date")
    .write
    .mode("append")
    .partitionBy("order_date")
    .saveAsTable("main.silver.orders")
)

For Delta tables on Databricks, compact with `OPTIMIZE`:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("optimize main.silver.orders where order_date >= date '2026-10-01'")

For Iceberg tables, rewrite data files from an Iceberg-enabled Spark catalog:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.rewrite_data_files(
  table => 'silver.orders'
)
""")

Real-world guidance: target file sizes are workload-dependent, but analytical tables often perform better with files in the hundreds of MB rather than thousands of tiny files.

Spark turns a scan into tasks using `spark.sql.files.maxPartitionBytes` (default 128 MB) and `spark.sql.files.openCostInBytes` (default 4 MB), which models the fixed cost of opening a file. When files are tiny, Spark packs many of them into one task, so the task count stays reasonable, but the per-file cost of listing, opening, reading footer metadata, and closing dominates. On table formats the damage is worse: the transaction log and metadata grow with file count, so planning time itself rises. Delta also carries per-file statistics that must be read and pruned, so many small files make data skipping slower as well as less effective.


Worked example. A streaming sink writes 500,000 files of about 1 MB each, for 500 GB. A scan must list 500,000 files and open each one. Compacting to 1 GB files gives about 500 files, so the open overhead drops by a factor of 1,000 while the bytes read stay the same. The target range most analytical tables settle on is 128 MB to 1 GB per file.


The common mistake is `coalesce(1)` to reduce file count, which serializes the write onto one core. Another is rewriting a whole table to fix one recent partition when a targeted `OPTIMIZE ... WHERE` does the job. A third is leaving write-time compaction off: `delta.autoOptimize.optimizeWrite` and `delta.autoOptimize.autoCompact` prevent the problem instead of repairing it afterwards.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Prevent small files at write time, then compact what already exists.
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

spark.sql("OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01'")


## How do you tune partition pruning and predicate pushdown?

Partition pruning skips whole table partitions. Predicate pushdown skips data inside files when file statistics prove rows cannot match.

Good filter:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders").where(
    "order_date >= date '2026-10-01' and order_date < date '2026-11-01'"
)

Bad filter that can block pruning:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders").where(
    "date_format(order_date, 'yyyy-MM') = '2026-10'"
)

The bad version wraps the partition column in a function. Spark may not be able to prune partitions efficiently.

Check the plan:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders.explain("formatted")

Look for `PartitionFilters` and `PushedFilters`.

For Unity Catalog tables, keep frequently filtered columns well-modeled. For Delta, consider clustering or liquid clustering where appropriate. For Iceberg, use hidden partition transforms such as `days(event_ts)` or `bucket(32, customer_id)`.

There are two different mechanisms and it helps to keep them apart. Partition pruning works on the directory layout: when a table is written with `partitionBy("order_date")`, the catalog knows the values that each directory holds, and the plan lists them under `PartitionFilters`. Data skipping works inside files: Parquet stores per-row-group min/max and null counts, and Delta stores per-file statistics in its transaction log, so a predicate can discard a whole row group or file without reading the data. Spark reports the ones it pushed to the reader under `PushedFilters`. Delta indexes statistics for the leading 32 columns by default, controlled by `delta.dataSkippingNumIndexedCols`.


Worked example. Five years of daily data partitioned by day gives about 1,825 directories. A query for one day prunes to roughly 1/1825 of the files. The same query on an unpartitioned table must consider every file, even though only a tiny fraction of rows match. On the file level, a 128 MB Parquet file with 1 MB row groups lets Spark skip most row groups when the predicate column is well clustered within the file.


The common mistake is a predicate that hides the literal from the optimizer. `F.date_format("order_date", "yyyy-MM") == "2026-10"`, `F.cast("order_date", "string").like("2026-10%")`, `F.lower("country") == "us"`, and `col.like("%suffix")` all prevent pruning or data skipping. Keep the predicate on the raw column and compare like with like. A type mismatch, such as comparing a timestamp column to a string literal with implicit conversion, has the same effect.


## How do you reduce data before expensive operations?

Filter rows and select columns before joins, aggregations, and window functions.

Better:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = (
    spark.table("main.silver.orders")
    .where("order_date >= date '2026-10-01'")
    .select("order_id", "customer_id", "amount", "order_date")
)

customers = (
    spark.table("main.gold.dim_customer")
    .select("customer_id", "customer_segment")
)

result = orders.join(customers, "customer_id").groupBy("customer_segment").sum("amount")

Worse:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
result = (
    spark.table("main.silver.orders")
    .join(spark.table("main.gold.dim_customer"), "customer_id")
    .where("order_date >= date '2026-10-01'")
    .groupBy("customer_segment")
    .sum("amount")
)

Spark can sometimes push filters and prune columns automatically, but clear code makes intent obvious and helps avoid blockers such as UDFs, complex expressions, or non-deterministic filters.

Catalyst already pushes filters and projections toward the scan, so explicit early filtering is partly about making that possible rather than doing it by hand. The optimizer cannot push a predicate through an aggregation, a window, a `distinct`, or an opaque expression, and it cannot push through a Python UDF at all. Columnar storage rewards projection: Parquet and Delta read only the column chunks a query needs, so selecting 4 columns from a 60-column table can reduce I/O by more than an order of magnitude even when the row count is unchanged.


Worked example. Orders has 60 columns and 2 billion rows. A report needs `order_id`, `customer_id`, `amount`, and `order_date`. Selecting those four before the join reads four column chunks instead of sixty. Filtering to one month first reduces rows by roughly 12x for a year of data. Together those two lines of code change the join input far more than any executor setting.


The common mistake is to write the `where` clause after the join and assume Spark will always reorder it. It usually can, but only for deterministic predicates; `rand()` or a UDF blocks the reordering. The other mistake is selecting `*` in a wide table out of habit, which forces the scan to read columns that are then discarded.


## How do you avoid driver memory problems?

The driver coordinates the job. It should not hold the full dataset.

Dangerous patterns:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
rows = df.collect()
pdf = df.toPandas()
ids = [row["customer_id"] for row in df.select("customer_id").collect()]

Safer alternatives:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
df.limit(20).show(truncate=False)
df.write.mode("overwrite").saveAsTable("main.tmp.debug_sample")

If you need a Python list for a small dimension, limit it intentionally:

In [ ]:
top_ids = [
    row["customer_id"]
    for row in df.select("customer_id").limit(1000).collect()
]

Interview point: driver memory failures often come from using Spark like pandas. Keep large data distributed, write results to tables, and collect only bounded samples.

The driver holds the query plan, the task scheduler's bookkeeping, any broadcast values, and every result that is collected back. `collect()` and `toPandas()` pull the entire result into that one JVM. `spark.driver.maxResultSize` (default 1 GB) caps the total size of results returned to the driver; exceeding it fails the job with a clear error, and the driver heap default of 1 GB often fails first. `toPandas` moves data through Arrow, which is efficient per row but does not change the fact that all of it must fit in driver memory.


Worked example. A table with 50 million rows at roughly 200 bytes per row is about 10 GB. `df.collect()` on it exceeds `spark.driver.maxResultSize` and the driver heap at once. Replacing it with `df.agg(F.sum("amount")).first()` returns a single row, and the aggregation runs distributed. If you genuinely need a sample, `df.limit(1000).toPandas()` bounds exactly what crosses the network.


The common mistake is using Spark like pandas: collecting ids to build a filter list, calling `toPandas()` for a "quick look", or returning a large Python list from a driver-side loop. Larger mistakes come from broadcasting something huge, because the broadcast value lives on the driver too. Keep large data distributed and collect only bounded results.


## How do you tune executor memory and cores?

Executor tuning balances CPU parallelism, memory per task, and cluster utilization. More memory is not always better; too many cores per executor can make each executor run many memory-hungry tasks at once.

Example job configuration:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark = (
    SparkSession.builder
    .appName("large-orders-aggregation")
    .config("spark.executor.instances", "20")
    .config("spark.executor.cores", "4")
    .config("spark.executor.memory", "16g")
    .config("spark.executor.memoryOverhead", "4g")
    .getOrCreate()
)

What to watch:

+ CPU underutilized: not enough partitions or too few cores
+ Frequent GC: executor memory pressure
+ Disk spill: shuffle or aggregation does not fit memory
+ Container killed: memory overhead too low
+ Slow tasks with no spill: skew or remote I/O

Real-world guidance: start with moderate cores per executor, enough memory overhead for Python and off-heap operations, and enough partitions to keep all cores busy. Then tune from Spark UI evidence.

An executor JVM is laid out in layers. A 300 MB reserved area is taken first and is not tunable. User memory is the remainder outside the Spark memory pools. The unified region covers execution memory (shuffles, sorts, joins, aggregations) and storage memory (cached blocks), sized by `spark.memory.fraction`, which defaults to 0.6 of the heap after the reserved area; `spark.memory.storageFraction` (default 0.5) is the share that storage may hold without being evicted by execution. On top of the heap sits `spark.executor.memoryOverhead`, off-heap memory for Python workers, Netty buffers, and JVM internals, defaulting to `max(384 MB, 0.10 * heap)`. The container size is heap plus overhead, which is what the cluster manager actually allocates.


Worked example. `spark.executor.memory=16g` gives overhead of `max(384 MB, 1.6 GB) = 1.6 GB`, so each container requests 17.6 GB. The unified region is `0.6 * (16 - 0.3) = 9.4 GB`, split roughly evenly between execution and storage until one needs more. With `spark.executor.cores=5`, five tasks share that 9.4 GB, so each task's execution budget is under 2 GB; a shuffle partition much larger than that will spill.


The common mistake is a very high core count per executor, such as 16, which concentrates many memory-hungry tasks in one JVM, produces long GC pauses, and can saturate the disk or object-store client. The opposite mistake is many one-core executors: each has its own overhead and its own copy of any broadcast, which wastes memory and adds JVM startup cost. Three to five cores per executor is the usual starting point. A third mistake is forgetting memory overhead entirely, which shows up as containers killed by the cluster manager rather than a clean Spark error.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# A static allocation with a memory layout you can reason about.
spark.conf.set("spark.executor.cores", "5")
spark.conf.set("spark.executor.memory", "16g")
spark.conf.set("spark.executor.memoryOverhead", "2g")
spark.conf.set("spark.executor.instances", "20")
spark.conf.set("spark.memory.fraction", "0.6")
spark.conf.set("spark.memory.storageFraction", "0.5")
print("container memory = 16g heap + 2g overhead = 18g")


## How do you tune Structured Streaming jobs?

Streaming tuning focuses on input rate, state size, trigger interval, checkpoint health, and sink throughput.

Example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("maxFilesPerTrigger", "500")
    .schema("event_id string, user_id string, event_ts timestamp")
    .load("s3://company-raw/events/")
)

query = (
    events
    .withWatermark("event_ts", "30 minutes")
    .dropDuplicates(["event_id"])
    .writeStream
    .option("checkpointLocation", "s3://company-checkpoints/events/")
    .trigger(processingTime="5 minutes")
    .toTable("main.silver.events")
)

Tuning levers:

+ Increase trigger interval if each batch cannot finish before the next one starts.
+ Use watermarks to bound state.
+ Control ingestion with `maxFilesPerTrigger` or source-specific rate limits.
+ Avoid expensive `foreachBatch` logic that scans large target tables each batch.
+ Keep checkpoint paths stable and isolated per stream.

Real-world example: a stream that deduplicates by `event_id` without a watermark can keep state forever. The job may run well for a week and then slow down as state grows.

Structured Streaming runs a micro-batch loop. Each trigger starts a normal Spark job for the data available at that moment, and the checkpoint directory records offsets, commit logs, and the state store. Stateful operators such as `dropDuplicates` or `flatMapGroupsWithState` keep their state in a versioned store inside the checkpoint, and a watermark bounds how far back that state must be retained. Input size per batch is bounded by source options such as `maxOffsetsPerTrigger` for Kafka and `maxFilesPerTrigger` for file sources; the trigger interval decides how often the loop runs.


Worked example. A stream consumes from Kafka with a 60-second trigger. Each batch starts to take 90 seconds because the topic grew. The next batch then starts late with even more data, and latency climbs without bound. The fix is to cap `maxOffsetsPerTrigger` so each batch is bounded, and widen the trigger to 2 minutes so batches do not overlap. Throughput settles and latency stops growing even though each batch is bigger.


The common mistake is stateful dedup without a watermark. State grows forever, the job looks fine for a week, then slows down as the state store grows and every batch reads and writes more of it. Two more: moving the checkpoint location, which orphans the state and offsets, and putting an expensive full-table scan inside `foreachBatch`, which repeats on every micro-batch.


## What Spark tuning checklist should you use in interviews?

Use this checklist when an interviewer asks, "How would you optimize a slow Spark job?"

+ First inspect the Spark UI and physical plan.
+ Check input size, file count, and partition pruning.
+ Filter rows and select columns early.
+ Avoid Python UDFs unless necessary.
+ Tune joins: broadcast small dimensions, deduplicate dimensions, handle skew.
+ Tune shuffle partitions and enable AQE.
+ Use `repartition` for balancing, `coalesce` for reducing output partitions.
+ Use salting for verified hot keys.
+ Cache only reused expensive DataFrames.
+ Avoid `collect()` and `toPandas()` on large data.
+ Compact small files and write sensible file sizes.
+ Validate improvements with runtime, shuffle metrics, spill metrics, and output correctness.

Strong interview answer: "I would not start by changing executor memory. I would inspect the plan and Spark UI, identify whether the bottleneck is scan, shuffle, skew, join, spill, or output files, then apply the smallest targeted fix and measure again."

Map the symptom to the first thing you would check, before you touch a config:


| Symptom | First check | Typical fix |
| --- | --- | --- |
| One task far slower than the rest | Task duration max vs median | Salt the key, repartition, AQE skew join |
| Stage time dominated by exchange | Shuffle read/write bytes | Broadcast, pre-aggregate, filter earlier |
| Spill (memory) and Spill (disk) above zero | Partition size and memory | More partitions, smaller advisory size, more memory |
| High GC time | Object churn | Kryo, serialized persist, fewer cached rows |
| Input tasks tiny, planning slow | File count | Compact with `OPTIMIZE`, tune write layout |
| Executor killed | Container memory | Raise `spark.executor.memoryOverhead`, lower cores |
| Driver out-of-memory | `collect` or broadcast | Remove collection, lower broadcast threshold |
| Many tiny output files | Output partition count | `coalesce` or `repartition` before writing |


The interview point is the order of operations: read the plan and the UI, form one hypothesis, change one thing, and measure. Naming a config is much weaker than describing the evidence that would justify it.


## How do you diagnose a slow Spark job?


The Spark UI is the instrument. Open the application, go to the Stages tab, and sort by duration to find the stage that owns the wall clock. Then open that stage and look at the task duration distribution, the shuffle counters, and the spill counters. Read the tree bottom-up: a parent stage's duration includes its children, so the interesting node is usually the leaf that feeds everything. The Jobs tab tells you which action triggered what; the SQL tab shows the same plan you would get from `explain("formatted")` but annotated with runtime row counts and shuffle sizes.


The single most informative signal is the shape of task durations within a stage. Spark finishes a stage when the last task finishes. If the median task takes 20 seconds and the maximum takes 12 minutes, the stage is skewed or badly partitioned, and adding executors will not help. If every task takes minutes and they all spill, the partitions are too large. If tasks are milliseconds and there are 50,000 of them, the job is over-partitioned and pays scheduling overhead rather than doing work.


Use this table to move from a metric to a cause and a fix:


| Metric you see | Likely cause | Fix |
| --- | --- | --- |
| Task duration max much greater than median | Skew or one oversized partition | Salt the hot key, repartition, enable AQE skew join |
| Shuffle read or write bytes large | A wide transformation | Broadcast the small side, pre-aggregate, filter earlier, change the key |
| Spill (memory) and Spill (disk) above zero | Partition does not fit execution memory | More shuffle partitions, smaller advisory size, more executor memory |
| GC time a large fraction of task time | Too many Java objects | Kryo serializer, serialized persist levels, fewer cached rows |
| Many tiny tasks, CPU mostly idle | Over-partitioning | Reduce partitions, let AQE coalesce |
| A few tasks with no spill, long scheduler delay | Under-partitioning or driver pressure | More partitions, fewer tasks per stage |
| Input tasks tiny while listing is slow | Small files | Compact with `OPTIMIZE`, tune write layout |
| Executor lost or container killed | Off-heap or overhead exhaustion | Raise `spark.executor.memoryOverhead`, lower cores per executor |
| Driver out-of-memory | `collect`, `toPandas`, or a large broadcast | Remove the collection, lower the broadcast threshold |
| Fetch failures and re-run map stages | Shuffle files evicted or executor lost | Increase shuffle service availability, reduce concurrent stages |


For the plan itself, `df.explain("formatted")` gives a readable operator tree, and `df.explain("cost")` shows the candidate plans with estimated sizes, which is how you learn why Spark chose a sort-merge join instead of a broadcast. `df.explain("codegen")` shows the generated Java for a fused stage. None of these are runtime facts; only the UI or the event log are.


The UI is not always available after the fact, so production jobs should write an event log. Set `spark.eventLog.enabled=true` and `spark.eventLog.dir` to a durable location, and point a history server at it with `spark.history.fs.logDirectory`. Event logs are newline-delimited JSON, so you can also parse them directly to extract per-stage metrics without any UI at all.


In [ ]:
import json
import tempfile
from collections import defaultdict
from pathlib import Path


def stage_metrics(event_log_path):
    """Sum shuffle and spill bytes per stage from a Spark event log."""
    totals = defaultdict(lambda: {"shuffle_read": 0, "shuffle_write": 0,
                                  "spill_disk": 0, "spill_mem": 0, "tasks": 0})
    with open(event_log_path) as handle:
        for line in handle:
            event = json.loads(line)
            if event.get("Event") != "SparkListenerTaskEnd":
                continue
            metrics = event.get("Task Metrics") or {}
            read = metrics.get("Shuffle Read Metrics") or {}
            write = metrics.get("Shuffle Write Metrics") or {}
            stage = totals[event.get("Stage ID")]
            stage["tasks"] += 1
            stage["shuffle_read"] += read.get("Total Bytes Read", 0)
            stage["shuffle_write"] += write.get("Shuffle Bytes Written", 0)
            stage["spill_disk"] += metrics.get("Disk Bytes Spilled", 0)
            stage["spill_mem"] += metrics.get("Memory Bytes Spilled", 0)
    return dict(totals)

# A minimal two-event sample so the parser runs without a cluster.
sample = [
    {"Event": "SparkListenerTaskEnd", "Stage ID": 3,
     "Task Metrics": {"Shuffle Read Metrics": {"Total Bytes Read": 5_000_000_000},
                      "Shuffle Write Metrics": {"Shuffle Bytes Written": 2_000_000_000},
                      "Disk Bytes Spilled": 0, "Memory Bytes Spilled": 1_000_000_000}},
    {"Event": "SparkListenerTaskEnd", "Stage ID": 3,
     "Task Metrics": {"Shuffle Read Metrics": {"Total Bytes Read": 1_000_000_000},
                      "Shuffle Write Metrics": {"Shuffle Bytes Written": 500_000_000},
                      "Disk Bytes Spilled": 250_000_000, "Memory Bytes Spilled": 0}},
]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "eventlog_v2_app-sample"
    path.write_text("\n".join(json.dumps(event) for event in sample), encoding="utf-8")
    for stage_id, metrics in stage_metrics(path).items():
        print(f"stage {stage_id}: {metrics}")

# In production point it at the real log:
# stage_metrics("/mnt/spark-events/app-20261007-0001")


The common mistake is to open the UI, see that a stage is slow, and immediately raise executor memory. The UI tells you which resource was exhausted, if any. Spill points at memory or partition size, skew points at the key, a large shuffle points at the plan, and tiny tasks point at partitioning. Diagnose the class of problem first, then pick the lever.


## How do you size executors, memory and parallelism?


The sizing model has three independent numbers: how many cores each executor gets, how much memory each executor gets, and how many executors run. Cores set the number of task slots, so total parallelism is `executors x cores_per_executor`. Memory sets how large each task's working set may be, since all tasks in an executor share one unified memory region. The number of shuffle partitions must be at least the number of task slots, or some cores sit idle; ideally it is a small multiple so a straggler does not leave the cluster empty at the end of a stage.


An executor exposes `spark.executor.cores` task slots. All of those tasks draw from the same unified region, so a 16 GB executor with 5 cores gives each concurrent task roughly 1.8 GB of execution memory, not 16 GB. That is why increasing cores without increasing memory can turn a job that did not spill into one that spills: more concurrent tasks, each with a smaller share.


`spark.executor.memory` is only the JVM heap. `spark.executor.memoryOverhead` is off-heap memory for Python worker processes, Netty, and JVM internals, and it defaults to `max(384 MB, 0.10 * spark.executor.memory)`. The cluster manager allocates heap plus overhead as the container, so a job with `spark.executor.memory=32g` actually requests about 35.2 GB per executor. When a container is killed by the cluster manager rather than failing with a Spark error, overhead is usually the culprit, and Python-heavy jobs need more of it.


Static allocation fixes `spark.executor.instances` at submit time and is predictable, which suits a stable nightly job. Dynamic allocation sets `spark.dynamicAllocation.enabled=true` and lets Spark add and remove executors between `minExecutors` and `maxExecutors`, starting from `initialExecutors`; with `spark.dynamicAllocation.shuffleTracking.enabled=true` it can release executors even without an external shuffle service. Dynamic allocation is a poor fit for very short jobs because the ramp-up time dominates, and a poor fit when many jobs share a cluster and would fight over the same idle resources.


A few large executors usually beat many tiny ones. Every executor is a JVM with a fixed overhead and a startup and warm-up cost, and every executor holds its own copy of any broadcast relation. Splitting the same total memory into 100 one-core executors rather than 20 five-core executors means five times as many JVMs, five times as many broadcast copies, and five times the scheduling bookkeeping, for exactly the same number of task slots. The counter-pressure is that very large executors create long GC pauses that stall every task in the JVM at once, so the usual landing zone is three to five cores and 8 to 32 GB of heap per executor.


The driver is separate. It holds the plan, the scheduler structures, broadcast values, and collected results, and it defaults to 1 GB of heap in many setups. A job that broadcasts a 500 MB dimension needs the driver to hold that 500 MB, so `spark.driver.memory` must be raised alongside the broadcast. Driver memory is not part of executor sizing and is easy to forget.


Worked example. A cluster has 10 nodes with 32 cores and 256 GB of RAM each. Leaving one core and 1 GB per node for the operating system and the NodeManager gives 31 cores and 255 GB per node. With 5 cores per executor that is `floor(31 / 5) = 6` executors per node, using 30 cores and leaving one idle, so 60 executors in total. Each gets `255 / 6 = 42.5`, so about 42 GB, and with the dynamic overhead formula that leaves roughly 38 GB of heap once overhead is subtracted. This is the standard cluster sizing arithmetic, and the leftovers are exactly why you should not expect 100 percent utilization.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Static sizing for a stable nightly job.
spark.conf.set("spark.executor.cores", "5")
spark.conf.set("spark.executor.memory", "38g")
spark.conf.set("spark.executor.instances", "60")

# Dynamic sizing for a bursty shared job.
spark.conf.set("spark.dynamicAllocation.enabled", "true")
spark.conf.set("spark.dynamicAllocation.minExecutors", "4")
spark.conf.set("spark.dynamicAllocation.maxExecutors", "60")
spark.conf.set("spark.dynamicAllocation.initialExecutors", "8")
spark.conf.set("spark.dynamicAllocation.shuffleTracking.enabled", "true")

# Keep shuffle parallelism at a small multiple of total task slots.
spark.conf.set("spark.sql.shuffle.partitions", "1200")


The common mistake is to size by total memory alone. A 256 GB node described as "32 cores and 256 GB" invites the assumption that one executor should use all of it, which produces a 32-core JVM with unpredictable GC and disk contention. The other common mistake is forgetting that overhead is added to the heap when the container is requested, so a job can fail to schedule because the requested container is larger than the node's remaining memory even though the heap would have fit.


## How do you handle spilling and memory pressure?


Spilling is the execution memory pool overflowing. Within a task, a sort, a hash aggregate, or a hash join allocates a large structure in execution memory. When the pool is exhausted, Spark does not fail; the `Spillable` structure writes sorted runs to local disk and reads them back during a merge or a probe. That is graceful degradation, but it converts memory-bound work into disk-bound work and adds serialization on both sides. Spilling is per task, so a stage can show spill on a handful of tasks while the other 999 are fine, which is itself a skew signal.


The cause is almost always partition size relative to per-task memory. If a shuffle partition is 3 GB and five tasks share an executor with 8 GB of unified memory, each task has roughly 1 GB and will spill. The fix can come from either side: make partitions smaller, or give each task more memory. The first is usually cheaper and safer.


See it in the UI as `Spill (Memory)` and `Spill (Disk)` on the stage page, and in the SQL tab as `spark.sql.execution.sort.spillSize` and the aggregate metrics attached to each operator. At the event-log level the task metrics are `Memory Bytes Spilled` and `Disk Bytes Spilled`. A large gap between memory spill and disk spill is normal: memory spill counts the bytes that were held in memory before being written, and disk spill counts what actually reached disk.


Fixes, roughly in order of preference:


+ Increase the shuffle partition count so each partition is smaller, or lower `spark.sql.adaptive.advisoryPartitionSizeInBytes`.
+ Filter and aggregate early so less data reaches the sort or join.
+ Remove an unnecessary `orderBy` or `sortWithinPartitions`; sorting is what forces the spillable structure to exist.
+ Prefer streaming aggregations over collect-then-aggregate patterns, because they hold less state.
+ Broadcast a small side instead of shuffling and hash-building a large one.
+ Only then add executor memory, and expect it to help only if the partition size is genuinely close to the limit.


The common mistake is treating spill as a pure memory problem. If one partition holds 10 percent of a 2 TB shuffle, no realistic amount of executor memory fixes it; the partition is the problem. The second mistake is to cache more data to "avoid recomputation" while the job is already spilling, which pushes execution memory out of the unified region and makes the spill worse.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Reduce partition size rather than adding memory.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
spark.conf.set("spark.sql.shuffle.partitions", "8000")

# If a single operator still spills, give the executors more memory instead.
spark.conf.set("spark.executor.memory", "24g")
spark.conf.set("spark.executor.memoryOverhead", "3g")


## How do you tune serialization, caching and GC?


Serialization is how Spark converts records to bytes for a shuffle, a broadcast, or a cache write. The default is `org.apache.spark.serializer.JavaSerializer`, which is general but slow and produces large output. `org.apache.spark.serializer.KryoSerializer` is faster and typically two to ten times more compact, at the cost of requiring registration for the best performance and of not being a drop-in for every class. Set `spark.serializer=org.apache.spark.serializer.KryoSerializer`; if you turn on `spark.kryo.registrationRequired=true`, every class must be registered via `spark.kryo.registrator` or the job fails loudly rather than silently falling back. Tune the Kryo buffer with `spark.kryoserializer.buffer.max` (default 64 MB) if you hit buffer-full errors on wide rows.


Cache levels are a memory-versus-CPU choice. `MEMORY_ONLY` stores deserialized JVM objects: fastest to read, largest in memory, and evicted partitions are recomputed. `MEMORY_AND_DISK` spills evicted blocks to local disk. `MEMORY_AND_DISK_SER` stores serialized bytes, which is much smaller and often the right default for a large cached frame, but every read pays deserialization. `DISK_ONLY` always writes to disk and is a last resort. The `_2` variants replicate each block to two executors, so they double storage for fault tolerance. `StorageLevel` also exposes `off_heap=True`, which moves cached blocks outside the JVM heap where they do not create garbage-collection pressure.


Object overhead is the reason serialized caching helps so much. A Java row is a collection of objects linked by references: an `InternalRow` or a case class has an object header, field references, and a boxed value per field. A record that is 30 bytes of logical data can occupy several hundred bytes in a deserialized cache. Serializing it stores close to the logical size, which is why a frame that will not fit `MEMORY_ONLY` often fits `MEMORY_AND_DISK_SER`.


For garbage collection, Spark 3 runs on Java 8 and later where G1 is available and is the usual choice; `spark.executor.extraJavaOptions="-XX:+UseG1GC"` makes it explicit. The goal is to keep GC time a small percentage of task time. High GC time usually means too many live objects, which points at oversized cached frames, very wide rows, or too much memory assigned to a single executor. Raising full-GC parallelism (`-XX:ParallelGCThreads`) helps on large heaps only when the heap itself is the problem. Off-heap caching with `spark.memory.offHeap.enabled` and `spark.memory.offHeap.size` moves cached bytes out of the heap entirely, which removes their GC cost.


The common mistake is changing the serializer and expecting a speedup on a job that is dominated by scans or by one skewed task. Serialization helps shuffles, broadcasts, and caches; it does nothing for I/O-bound scans. The second mistake is using `_SER` levels for a small frame that is read repeatedly, where the deserialization cost outweighs the memory saving. The third is turning on `registrationRequired` before registering classes, which fails the job immediately.


In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark import StorageLevel

# Compact shuffle and cache serialization.
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
spark.conf.set("spark.kryoserializer.buffer.max", "128m")

base = (
    spark.table("main.silver.orders")
    .where("order_date >= date '2026-10-01'")
    .select("order_id", "customer_id", "amount", "order_date")
    .persist(StorageLevel.MEMORY_AND_DISK_SER)
)
base.count()  # materialize the cache with a cheap action

# Move cached bytes off the Java heap if GC time is high.
spark.conf.set("spark.memory.offHeap.enabled", "true")
spark.conf.set("spark.memory.offHeap.size", "8g")


## How do you tune Delta and file layout for performance?


File layout is usually the highest-leverage tuning lever for a table that is read repeatedly, because it changes the cost of every future query rather than the cost of one job. The target file size for analytical tables is roughly 128 MB to 1 GB: large enough that per-file overhead is negligible, small enough that a single file can still be read in parallel and rewritten cheaply. Delta's `delta.targetFileSize` and Databricks' `delta.tuneFileSizesForRewrites` guide `OPTIMIZE`, and write-time compaction with `delta.autoOptimize.optimizeWrite` and `delta.autoOptimize.autoCompact` prevents small files from accumulating in the first place. Small files hurt twice: scans open more files, and the transaction log carries more per-file statistics, so planning time grows too.


`OPTIMIZE` rewrites many small files into fewer target-sized files. `ZORDER BY (col)` additionally co-locates rows with similar values across the files it writes, which improves data skipping for the z-ordered columns; the effect is strongest for the first column and decays after a few columns, and it is a one-time layout that degrades as new data arrives. Liquid clustering, declared with `CLUSTER BY (col, ...)`, is the newer approach: the table records the clustering keys in metadata, and `OPTIMIZE` incrementally re-clusters only the data that needs it rather than rewriting everything. On a liquid-clustered table you run `OPTIMIZE` normally, or `OPTIMIZE ... FULL` to force a full recluster; you cannot combine liquid clustering with `partitionBy` or with `ZORDER` on the same table.


Partition pruning and data skipping are the mechanisms these layouts serve. A partition column prunes whole directories at plan time; z-order or liquid clustering improves the min/max ranges inside files so that file and row-group statistics can discard more of them. A predicate on a column that is not in the clustering keys and not in the leading `delta.dataSkippingNumIndexedCols` columns cannot skip much. This is why the same two-column key can be worth more than doubling the cluster size: a scan that reads 2 percent of the files finishes before a bigger cluster has finished reading the other 98 percent.


`VACUUM` removes data files that are no longer referenced by the transaction log and are older than the retention period, which defaults to 7 days. Vacuuming is not a performance operation; it reclaims storage and shortens file listing. Setting a shorter retention period than the default trades Time Travel history for space and can break readers that are still using an older snapshot, so on a table with concurrent writers or long-running readers keep the default and let the platform manage it.


The common mistake is optimizing on a schedule instead of on evidence. Running `ZORDER` every night on a table that only receives appended data rewrites the same files repeatedly for no pruning benefit. The second mistake is vacuuming aggressively to "speed up" a table, which removes Time Travel history and can strand a long-running job. The third is partitioning on a high-cardinality column such as `customer_id`, which produces millions of tiny directories and defeats the purpose.


In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Rewrite small files and cluster for skipping (Delta with Z-order).
spark.sql("""
OPTIMIZE main.silver.orders
WHERE order_date >= date '2026-10-01'
ZORDER BY (customer_id)
""")

# Liquid clustering: declare the keys once, then OPTIMIZE incrementally.
spark.sql("ALTER TABLE main.silver.orders CLUSTER BY (order_date, customer_id)")
spark.sql("OPTIMIZE main.silver.orders")

# Reclaim storage from files no longer in the log.
spark.sql("VACUUM main.silver.orders RETAIN 168 HOURS")


```text
small files:  120,000 files x 3 MB   -> slow listing, more per-file stats, weak skipping
target files:     400 files x 900 MB -> one task per file, good skipping, cheap rewrites

layout beats cluster size when the filter is selective:
  clustered table, 1 day filter  -> reads  2% of files
  unclustered table, bigger cluster -> reads 100% of files, faster per file, still 50x more I/O
```


## How do you tune Python UDFs and pandas UDFs?


A row-at-a-time Python UDF is the most expensive thing you can put in a Spark plan. Spark must serialize each row from the JVM to a Python worker process, run the Python function, and serialize the result back. The function is opaque to Catalyst, so it blocks predicate pushdown, constant folding, and whole-stage codegen, and it forces the plan to materialize rows at that operator. A built-in function runs inside the JVM on the generated code path and is typically one to two orders of magnitude faster for the same logic.


Vectorized UDFs change the cost model by amortizing the boundary crossing. A pandas UDF declared with `@pandas_udf` receives a `pandas.Series` or `DataFrame` per batch instead of a row, transfers the batch through Apache Arrow in a columnar format, and runs vectorized pandas or NumPy code. `mapInPandas` gives you an iterator of DataFrames and can change the shape of the data, which is what you use when one input batch produces zero, one, or many output batches. `applyInPandas` is the grouped form, running one function call per group, which is useful for per-group model scoring or per-group resampling. The batch size is controlled by `spark.sql.execution.arrow.maxRecordsPerBatch` (default 10,000), and Arrow-based transfer is enabled by `spark.sql.execution.arrow.pyspark.enabled`.


The order of preference is: use a built-in expression, then a SQL expression, then a vectorized pandas UDF, and only then a row UDF. Many things people write as UDFs already exist as functions: conditional logic is `F.when`, string parsing is `F.regexp_extract`, hashing is `F.xxhash64`, date math is `F.date_add`. A row UDF should be reserved for logic that genuinely cannot be expressed in those terms.


On Databricks, Photon is a native vectorized C++ query engine that replaces the JVM operators for scans, joins, aggregations, and expressions. It makes native expressions even faster and reduces the relative cost of the JVM path, but it does not run Python UDFs; a Python UDF in a Photon plan forces a transition back to the JVM path, and the surrounding work may not be accelerated at all. Code that avoids Python UDFs therefore benefits from Photon twice: once because the operators are faster, and once because the plan stays entirely in the accelerated engine.


The common mistake is writing a UDF without checking whether a built-in exists. The second is using a row UDF where a vectorized one would do, which is the difference between a per-row Python call and a per-batch NumPy call. The third is assuming that a pandas UDF is free: it is much cheaper than a row UDF but still crosses the JVM boundary per batch, so a filter or aggregation that could have happened in the JVM before the UDF should still happen first.


In [ ]:
# ILLUSTRATIVE - requires pandas, pyspark: install to run, or read on.
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Preferred: a built-in expression.
df = df.withColumn("amount_abs", F.abs("amount"))

# Vectorized UDF: one call per Arrow batch, not one call per row.
@pandas_udf(DoubleType())
def zscore(values: pd.Series) -> pd.Series:
    return (values - values.mean()) / values.std()


df = df.withColumn("amount_z", zscore(F.col("amount")))


# Shape-changing work: an iterator of DataFrames in, DataFrames out.
def rank_batches(iterator):
    for pdf in iterator:
        pdf = pdf.copy()
        pdf["amount_rank"] = pdf["amount"].rank(pct=True)
        yield pdf


ranked = df.mapInPandas(
    rank_batches, schema="order_id long, amount double, amount_rank double"
)


## How do you measure whether a tuning change worked?


A tuning change is an experiment, so it needs a baseline and a comparison on the same data, the same code, and the same cluster. Before the change, record at least: wall-clock duration, shuffle bytes read and written, memory and disk spill, total task count and the max task duration, GC time, output file count, and the cost of the run. After the change, record exactly the same numbers. If you only remember the wall clock, you cannot tell whether the job got faster because the plan improved or because the cluster happened to be idle.


Shuffle bytes are the leading indicator because they predict the network, disk, and serialization work that follows. A change that halves shuffle bytes will almost always reduce runtime even if the wall clock looks noisy on the first run. Wall clock is what the business cares about but it is also the noisiest metric: caching, cluster contention, and object-store throttling all move it without any change to the plan. Compare medians over several runs rather than a single pair.


Cost is a separate axis from latency. A change that halves runtime by doubling the cluster size may be credit-neutral or a loss, depending on how the platform bills and whether the cluster is shared. The same query that runs 600 seconds on a small cluster and 300 seconds on a cluster twice the size consumes roughly the same compute. If the runtime does not fall by at least the factor by which the resource rate rose, the change costs more for the same work.


A short checklist for the comparison:


+ Did shuffle bytes read and written fall, and by how much?
+ Did spill go to zero, and did the max task duration approach the median?
+ Did the stage count or the task count change, and was that intended?
+ Did the wall clock improve across several runs, not just one?
+ Did the cost per run improve, stay flat, or get worse?
+ Is the output identical, byte for byte where it matters?
+ Did the change fix one job at the expense of another that shares the cluster?


The common mistake is to compare a before run on a busy Monday with an after run on a quiet Sunday, or to change two settings at once, which makes the result unattributable. The second mistake is to trust a single wall-clock measurement when the improvement is smaller than the run-to-run variance. If the change is smaller than the noise, run the baseline and the variant alternately several times.


In [ ]:
import json

# Record the same fields before and after, then compare like with like.
def summarize(before, after):
    def delta(key):
        b, a = before[key], after[key]
        change = (a - b) / b * 100 if b else 0.0
        return f"{b:,} -> {a:,} ({change:+.1f}%)"

    for key in ("wall_clock_s", "shuffle_read_bytes", "shuffle_write_bytes",
                "disk_bytes_spilled", "max_task_s", "task_count"):
        print(f"{key:22s} {delta(key)}")


baseline = {"wall_clock_s": 3600, "shuffle_read_bytes": 1_200_000_000_000,
            "shuffle_write_bytes": 1_200_000_000_000, "disk_bytes_spilled": 8_000_000_000,
            "max_task_s": 540, "task_count": 200_000}
tuned = {"wall_clock_s": 1_200, "shuffle_read_bytes": 20_000_000_000,
         "shuffle_write_bytes": 20_000_000_000, "disk_bytes_spilled": 0,
         "max_task_s": 95, "task_count": 40_000}

summarize(baseline, tuned)
print(json.dumps({"shuffle_reduction_pct": round(
    (1 - tuned["shuffle_read_bytes"] / baseline["shuffle_read_bytes"]) * 100, 1)}))


## Spark tuning exercises


These exercises use only the Python standard library and model Spark tuning arithmetic with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.


The exercises mirror the quantities you read from the Spark UI and from `explain`: shuffle bytes and partition sizes, broadcast thresholds, per-partition row counts, spill, and executor arithmetic. Treat the asserted numbers as a check on the formula rather than a prediction of a real cluster. The value is knowing which quantity to compute and which direction a change moves it.


### Try it 1. Choose shuffle partitions


**Task.** A shuffle moves 640 GB (binary units). Compute the number of shuffle partitions needed for a 128 MB target partition size, then for a 256 MB target.


**Hint.** `partitions = ceil(shuffle_bytes / target_bytes)`. One GB here is `1024` MB.


**Expected result.** `5,120` partitions for 128 MB and `2,560` for 256 MB.


In [ ]:
import math


def shuffle_partitions(shuffle_gb, target_mb):
    shuffle_mb = shuffle_gb * 1024
    return math.ceil(shuffle_mb / target_mb)


partitions_128 = shuffle_partitions(640, 128)
partitions_256 = shuffle_partitions(640, 256)
print(f"640 GB at 128 MB/partition -> {partitions_128:,} partitions")
print(f"640 GB at 256 MB/partition -> {partitions_256:,} partitions")

assert partitions_128 == 5120
assert partitions_256 == 2560
assert partitions_128 == 2 * partitions_256


### Try it 2. Broadcast threshold decision


**Task.** Using the default auto-broadcast threshold of 10 MB, decide whether a 7.5 MB dimension and a 45 MB dimension would be broadcast. Then show the driver and network cost of forcing the 45 MB dimension to broadcast across 60 executors.


**Hint.** Compare bytes against `10 * 1024 * 1024`. A forced broadcast replicates the relation to every executor.


**Expected result.** The 7.5 MB dimension broadcasts, the 45 MB one does not, and forcing it moves about `2,700` MB in total.


In [ ]:
MB = 1024 * 1024
THRESHOLD = 10 * MB
EXECUTORS = 60


def would_broadcast(size_mb):
    return size_mb * MB <= THRESHOLD


small, large = 7.5, 45.0
print(f"{small} MB -> broadcast: {would_broadcast(small)}")
print(f"{large} MB -> broadcast: {would_broadcast(large)}")

network_mb = large * EXECUTORS
print(f"forced broadcast network = {network_mb:,.0f} MB")

assert would_broadcast(small) is True
assert would_broadcast(large) is False
assert network_mb == 2700.0


### Try it 3. Detect skew from per-partition row counts


**Task.** Given the row counts of eight shuffle partitions, compute the median, the maximum, and the ratio of max to median. Flag the stage as skewed when the max exceeds five times the median.


**Hint.** Sort the counts and take the middle value, averaging the two middle values for an even count.


**Expected result.** Median `101`, maximum `5,000`, ratio about `49.5`, and the stage is skewed.


In [ ]:
counts = [100, 105, 98, 102, 100, 5000, 99, 103]


def median(values):
    ordered = sorted(values)
    mid = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2


med = median(counts)
peak = max(counts)
ratio = peak / med
skewed = ratio > 5

print(f"median = {med:.1f}, max = {peak}, ratio = {ratio:.1f}")
print("skewed:", skewed)

assert med == 101
assert peak == 5000
assert round(ratio, 1) == 49.5
assert skewed is True


### Try it 4. Salt a hot key and recompute the distribution


**Task.** One million rows are keyed by customer. The key `guest` holds 40 percent and four normal keys split the rest evenly. Compute the largest partition share before salting, then salt `guest` into 32 buckets evenly and recompute it.


**Hint.** After salting, the largest partition is either a salted bucket of the hot key or one of the normal keys, whichever is larger.


**Expected result.** The largest share falls from `0.40` to `0.15`, roughly a `2.67x` improvement.


In [ ]:
TOTAL = 1_000_000
SALT_BUCKETS = 32
hot = int(TOTAL * 0.40)
normal = (TOTAL - hot) / 4

before = hot / TOTAL
salted_bucket = hot / SALT_BUCKETS
after = max(salted_bucket, normal) / TOTAL

print(f"before: hot key share = {before:.2f}")
print(f"after : largest share = {after:.2f} (salted bucket = {salted_bucket:,.0f} rows)")
print(f"improvement = {before / after:.2f}x")

assert before == 0.40
assert after == 0.15
assert round(before / after, 2) == 2.67


### Try it 5. Decide repartition versus coalesce


**Task.** After a filter, a DataFrame has 1,000 partitions holding 4 GB in total and the downstream step wants 16 tasks. Decide between `coalesce(16)` and `repartition(16)` using a 128 MB target partition size. Repeat for a 512 MB result with 4 target tasks.


**Hint.** `coalesce` merges whole upstream partitions without a shuffle, so it is safe only when the merged partition fits the target. Otherwise you need `repartition`.


**Expected result.** 4 GB into 16 tasks is 256 MB each, so use `repartition`; 512 MB into 4 tasks is 128 MB each, so `coalesce` is fine.


In [ ]:
MB = 1024 * 1024


def choose_op(total_mb, target_tasks, target_mb=128):
    per_task = total_mb / target_tasks
    choice = "coalesce" if per_task <= target_mb else "repartition"
    return choice, per_task


choice_a, per_a = choose_op(4 * 1024, 16)
choice_b, per_b = choose_op(512, 4)
print(f"4096 MB / 16 = {per_a:.0f} MB per task -> {choice_a}")
print(f" 512 MB /  4 = {per_b:.0f} MB per task -> {choice_b}")

assert choice_a == "repartition"
assert choice_b == "coalesce"


### Try it 6. Estimate executor count from cores and memory


**Task.** A cluster has 10 nodes with 32 cores and 256 GB of RAM each. Leave one core and 1 GB per node for the operating system and the NodeManager. With 5 cores per executor, compute executors per node, total executors, total cores used, and the memory per executor.


**Hint.** `executors_per_node = floor((cores_per_node - 1) / cores_per_executor)`, then divide the usable memory by that count.


**Expected result.** `6` executors per node, `60` in total, `30` cores used per node, and about `42` GB of memory each.


In [ ]:
NODES = 10
CORES_PER_NODE = 32
MEM_PER_NODE_GB = 256
CORES_PER_EXECUTOR = 5

usable_cores = CORES_PER_NODE - 1
usable_mem = MEM_PER_NODE_GB - 1

executors_per_node = usable_cores // CORES_PER_EXECUTOR
cores_used = executors_per_node * CORES_PER_EXECUTOR
mem_per_executor = usable_mem // executors_per_node
total_executors = executors_per_node * NODES

print(f"executors per node = {executors_per_node}")
print(f"total executors    = {total_executors}")
print(f"cores used per node= {cores_used} of {CORES_PER_NODE}")
print(f"memory per executor= {mem_per_executor} GB")

assert executors_per_node == 6
assert total_executors == 60
assert cores_used == 30
assert mem_per_executor == 42


### Try it 7. Compute AQE coalescing of small partitions


**Task.** A shuffle produces 200 partitions of 2 MB each. AQE coalesces adjacent partitions toward `spark.sql.adaptive.advisoryPartitionSizeInBytes = 64m`. Compute the number of partitions after coalescing and the resulting average partition size.


**Hint.** `coalesced = ceil(total_bytes / advisory_bytes)`. Each coalesced partition packs whole input partitions, so the total is unchanged.


**Expected result.** `400` MB coalesces into `7` partitions of about `57.1` MB each.


In [ ]:
import math

INPUT_PARTITIONS = 200
PARTITION_MB = 2
ADVISORY_MB = 64

total_mb = INPUT_PARTITIONS * PARTITION_MB
coalesced = math.ceil(total_mb / ADVISORY_MB)
avg_mb = total_mb / coalesced

print(f"total = {total_mb} MB across {INPUT_PARTITIONS} partitions")
print(f"coalesced -> {coalesced} partitions, average {avg_mb:.1f} MB")

assert total_mb == 400
assert coalesced == 7
assert round(avg_mb, 1) == 57.1


### Try it 8. Compare two plans by estimated shuffle bytes


**Task.** Plan A is a sort-merge join that shuffles 1.2 TB in each direction. Plan B broadcasts a 300 MB dimension to 60 executors and shuffles the fact side not at all. Compare total bytes moved across the network and compute the reduction factor.


**Hint.** Plan A moves its shuffle write plus its shuffle read. Plan B moves the broadcast size once per executor. Convert everything to MB before comparing.


**Expected result.** Plan A moves `2.4` TiB, Plan B about `18,000` MB, a reduction of roughly `139.8x`.


In [ ]:
MB_PER_GIB = 1024

plan_a_write_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_read_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_total = plan_a_write_mb + plan_a_read_mb

broadcast_mb = 300
executors = 60
plan_b_total = broadcast_mb * executors

reduction = plan_a_total / plan_b_total
print(f"plan A moves {plan_a_total:,.0f} MB")
print(f"plan B moves {plan_b_total:,.0f} MB")
print(f"reduction = {reduction:.1f}x")

assert round(plan_a_total / (1024 * 1024), 1) == 2.4
assert plan_b_total == 18000
assert round(reduction, 1) == 139.8


## Real world use cases for Spark tuning

### 1. A nightly job that regressed from 20 minutes to 3 hours

The job was fine until one tenant's data grew. The Spark UI showed stage 7 with one task taking 90% of the time and shuffle spill - classic skew plus too many shuffle partitions.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Diagnosis: the physical plan and stage metrics told the story
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")

# Also stop forcing a huge fixed partition count
spark.conf.set("spark.sql.shuffle.partitions", "auto")

Why it worked: AQE coalesced the small post-shuffle partitions and split the skewed join, so runtime dropped back to minutes without changing executor memory at all.

### 2. A join on a hot customer key

One customer produced 40% of all rows, so salting is the targeted fix when AQE skew handling is not enough or the key is used in a non-join shuffle.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

salt_buckets = 16

large = orders.withColumn("salt", (F.rand() * salt_buckets).cast("int"))
dim = customers.crossJoin(spark.range(salt_buckets).withColumnRenamed("id", "salt"))

joined = large.join(F.broadcast(dim), ["customer_id", "salt"])

Why it worked: splitting the hot key into 16 sub-keys lets 16 tasks process it, turning a single straggler task into balanced work. Measure the key distribution first - salting adds a shuffle and is wasted effort if the skew is not real.

### 3. The small-files problem after streaming ingestion

A streaming sink wrote thousands of tiny files per hour. Queries slowed because planning and opening files dominated the scan.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Compact the partition into fewer, larger files
(
    spark.read.parquet("s3://acme-curated/orders/ds=2026-10-04/")
    .coalesce(8)
    .write.mode("overwrite")
    .parquet("s3://acme-curated/orders_compacted/ds=2026-10-04/")
)

# Prevent the problem at the source where possible
spark.conf.set("spark.sql.files.maxRecordsPerFile", 5_000_000)
spark.conf.set("spark.sql.files.openCostInBytes", 128 * 1024 * 1024)

Why it worked: fewer files meant fewer tasks and less per-file overhead. On a table format, run the platform compaction (`OPTIMIZE`, `rewrite_data_files`) instead of rewriting files yourself.

### 4. A broadcast join that stopped broadcasting

A "small" product dimension grew past the broadcast threshold, so every nightly run shuffled both sides.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Raise the threshold deliberately (and know the driver memory cost)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 64 * 1024 * 1024)

# Or force it when you know the relation is small
joined = events.join(F.broadcast(products), "product_id")

Why it worked: a broadcast avoids the shuffle entirely. The trade-off is driver memory and broadcast time - never broadcast a relation you have not checked the size of.

### 5. Streaming latency that kept climbing

A Structured Streaming job fell behind because each micro-batch tried to consume everything available, producing ever-larger batches.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("maxOffsetsPerTrigger", 500_000)     # bound each micro-batch
    .load()
)

(
    stream.writeStream
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="2 minutes")
    .start()
)

Why it worked: `maxOffsetsPerTrigger` caps batch size so each batch finishes before the next arrives, and the fixed trigger makes throughput predictable. Also check watermarking and state-store growth for stateful operators.

### 6. Driver OOM from collecting results

The driver died whenever a job called `collect()` or `toPandas()` on a large DataFrame.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Before: pulls every row into the driver
rows = df.collect()

# After: aggregate in the cluster and bring back one row
summary = df.agg(F.sum("amount").alias("total")).first()

# For small samples only, cap what crosses the network
sample = df.limit(1000).toPandas()

Why it worked: the driver is for coordination, not data. Aggregations and `limit` keep results inside the cluster, and only small results cross to the driver.

Across all six cases the pattern is the same: the UI or the plan named the bottleneck before anyone changed a setting. Two of them were fixed by configuration alone, one by rewriting a join, one by changing the write layout, and one by removing work from the driver. None of them was fixed by adding memory, and in every case the measurement that proved the fix was a byte count or a per-task duration rather than a feeling that the job seemed faster.


A useful habit when you present one of these in an interview is to state the before and after numbers and the mechanism that connects them. "Stage 7 had one task at 90 percent of the stage time and shuffle spill; AQE split the skewed partition and the spill went to zero, so runtime went from three hours to twenty minutes" is a complete answer. "I enabled AQE" is not.
